# Pairwise Post-Hoc — Wilcoxon Signed-Rank + Holm Correction

After Friedman rejects $H_0$, we run pairwise tests on the **same 16 blocks**
for two metrics:

| Metric | Blocks |
|---|---|
| **Accuracy** | 16 (4 datasets × 4 FS) |
| **G-Mean**   | 16 (4 datasets × 4 FS) |

For each of $\binom{8}{2}=28$ pairs $(A,B)$:
1. Wilcoxon signed-rank on paired differences $d_i = \text{score}_A(i) - \text{score}_B(i)$.
2. Sort all 28 p-values; Holm step-down: reject $H_0^{(i)}$ if $p_{(i)} < \alpha/(n_c - i)$.

> Embedded data — Run All in Colab.


## 1. Imports

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon
from itertools import combinations


## 2. Embedded scores

In [2]:
MODELS = ['LogisticReg', 'KNN', 'SVM-RBF', 'DecisionTree', 'RandomForest', 'GradBoost', 'XGBoost', 'Stacker']
FS_ORDER = ['Full', 'RFE', 'Lasso-LR', 'Forward']

# Accuracy: 16 blocks
ACC = {
    'Australian': {
        'Full': {'LogisticReg': 0.8333, 'KNN': 0.8551, 'SVM-RBF': 0.8333, 'DecisionTree': 0.8116, 'RandomForest': 0.8188, 'GradBoost': 0.8551, 'XGBoost': 0.8696, 'Stacker': 0.8623},
        'RFE': {'LogisticReg': 0.8333, 'KNN': 0.8261, 'SVM-RBF': 0.8333, 'DecisionTree': 0.7971, 'RandomForest': 0.8188, 'GradBoost': 0.8333, 'XGBoost': 0.8333, 'Stacker': 0.8333},
        'Lasso-LR': {'LogisticReg': 0.8333, 'KNN': 0.8043, 'SVM-RBF': 0.8333, 'DecisionTree': 0.8261, 'RandomForest': 0.8116, 'GradBoost': 0.8478, 'XGBoost': 0.8768, 'Stacker': 0.8406},
        'Forward': {'LogisticReg': 0.7971, 'KNN': 0.8116, 'SVM-RBF': 0.7826, 'DecisionTree': 0.8261, 'RandomForest': 0.8188, 'GradBoost': 0.8188, 'XGBoost': 0.8116, 'Stacker': 0.8116},
    },
    'Japanese': {
        'Full': {'LogisticReg': 0.8623, 'KNN': 0.8116, 'SVM-RBF': 0.8478, 'DecisionTree': 0.8841, 'RandomForest': 0.8768, 'GradBoost': 0.9058, 'XGBoost': 0.8986, 'Stacker': 0.8986},
        'RFE': {'LogisticReg': 0.8841, 'KNN': 0.8696, 'SVM-RBF': 0.7971, 'DecisionTree': 0.8768, 'RandomForest': 0.8986, 'GradBoost': 0.9058, 'XGBoost': 0.8768, 'Stacker': 0.9275},
        'Lasso-LR': {'LogisticReg': 0.8551, 'KNN': 0.8261, 'SVM-RBF': 0.8406, 'DecisionTree': 0.8261, 'RandomForest': 0.9130, 'GradBoost': 0.8913, 'XGBoost': 0.9203, 'Stacker': 0.8986},
        'Forward': {'LogisticReg': 0.8913, 'KNN': 0.8478, 'SVM-RBF': 0.8841, 'DecisionTree': 0.8261, 'RandomForest': 0.8841, 'GradBoost': 0.8913, 'XGBoost': 0.8913, 'Stacker': 0.9058},
    },
    'German': {
        'Full': {'LogisticReg': 0.6850, 'KNN': 0.6450, 'SVM-RBF': 0.7400, 'DecisionTree': 0.5850, 'RandomForest': 0.7100, 'GradBoost': 0.7450, 'XGBoost': 0.6750, 'Stacker': 0.7100},
        'RFE': {'LogisticReg': 0.6900, 'KNN': 0.6950, 'SVM-RBF': 0.7500, 'DecisionTree': 0.7250, 'RandomForest': 0.6900, 'GradBoost': 0.7700, 'XGBoost': 0.7500, 'Stacker': 0.7400},
        'Lasso-LR': {'LogisticReg': 0.7150, 'KNN': 0.6700, 'SVM-RBF': 0.7150, 'DecisionTree': 0.6800, 'RandomForest': 0.7100, 'GradBoost': 0.7350, 'XGBoost': 0.6950, 'Stacker': 0.7150},
        'Forward': {'LogisticReg': 0.7100, 'KNN': 0.7000, 'SVM-RBF': 0.7350, 'DecisionTree': 0.6350, 'RandomForest': 0.6650, 'GradBoost': 0.7100, 'XGBoost': 0.6800, 'Stacker': 0.7000},
    },
    'Taiwan': {
        'Full': {'LogisticReg': 0.8048, 'KNN': 0.7750, 'SVM-RBF': 0.8013, 'DecisionTree': 0.7973, 'RandomForest': 0.8003, 'GradBoost': 0.7962, 'XGBoost': 0.8007, 'Stacker': 0.8003},
        'RFE': {'LogisticReg': 0.7913, 'KNN': 0.7773, 'SVM-RBF': 0.7950, 'DecisionTree': 0.8087, 'RandomForest': 0.7698, 'GradBoost': 0.7967, 'XGBoost': 0.7970, 'Stacker': 0.7933},
        'Lasso-LR': {'LogisticReg': 0.8053, 'KNN': 0.7733, 'SVM-RBF': 0.8028, 'DecisionTree': 0.8057, 'RandomForest': 0.8025, 'GradBoost': 0.7902, 'XGBoost': 0.7988, 'Stacker': 0.7983},
        'Forward': {'LogisticReg': 0.7858, 'KNN': 0.7918, 'SVM-RBF': 0.7892, 'DecisionTree': 0.7840, 'RandomForest': 0.8053, 'GradBoost': 0.7888, 'XGBoost': 0.8023, 'Stacker': 0.8012},
    },
}

# G-Mean: 16 blocks
GMEAN = {
    'Australian': {
        'Full': {'LogisticReg': 0.8390, 'KNN': 0.8508, 'SVM-RBF': 0.8364, 'DecisionTree': 0.8087, 'RandomForest': 0.8241, 'GradBoost': 0.8598, 'XGBoost': 0.8724, 'Stacker': 0.8645},
        'RFE': {'LogisticReg': 0.8390, 'KNN': 0.8305, 'SVM-RBF': 0.8387, 'DecisionTree': 0.8013, 'RandomForest': 0.8236, 'GradBoost': 0.8382, 'XGBoost': 0.8374, 'Stacker': 0.8390},
        'Lasso-LR': {'LogisticReg': 0.8390, 'KNN': 0.8042, 'SVM-RBF': 0.8374, 'DecisionTree': 0.8271, 'RandomForest': 0.8170, 'GradBoost': 0.8521, 'XGBoost': 0.8803, 'Stacker': 0.8452},
        'Forward': {'LogisticReg': 0.8024, 'KNN': 0.8171, 'SVM-RBF': 0.7874, 'DecisionTree': 0.8316, 'RandomForest': 0.8241, 'GradBoost': 0.8218, 'XGBoost': 0.8159, 'Stacker': 0.8170},
    },
    'German': {
        'Full': {'LogisticReg': 0.6137, 'KNN': 0.5294, 'SVM-RBF': 0.6492, 'DecisionTree': 0.6298, 'RandomForest': 0.6882, 'GradBoost': 0.6170, 'XGBoost': 0.6330, 'Stacker': 0.6711},
        'RFE': {'LogisticReg': 0.6708, 'KNN': 0.6083, 'SVM-RBF': 0.6705, 'DecisionTree': 0.6835, 'RandomForest': 0.6581, 'GradBoost': 0.6294, 'XGBoost': 0.6990, 'Stacker': 0.6992},
        'Lasso-LR': {'LogisticReg': 0.6429, 'KNN': 0.5606, 'SVM-RBF': 0.6604, 'DecisionTree': 0.6294, 'RandomForest': 0.6544, 'GradBoost': 0.5928, 'XGBoost': 0.6583, 'Stacker': 0.6937},
        'Forward': {'LogisticReg': 0.6473, 'KNN': 0.6414, 'SVM-RBF': 0.6619, 'DecisionTree': 0.6245, 'RandomForest': 0.6448, 'GradBoost': 0.6882, 'XGBoost': 0.6234, 'Stacker': 0.6848},
    },
    'Japanese': {
        'Full': {'LogisticReg': 0.8612, 'KNN': 0.7785, 'SVM-RBF': 0.8445, 'DecisionTree': 0.8871, 'RandomForest': 0.8760, 'GradBoost': 0.8990, 'XGBoost': 0.8951, 'Stacker': 0.8951},
        'RFE': {'LogisticReg': 0.8857, 'KNN': 0.8513, 'SVM-RBF': 0.7863, 'DecisionTree': 0.8803, 'RandomForest': 0.8971, 'GradBoost': 0.9014, 'XGBoost': 0.8760, 'Stacker': 0.9224},
        'Lasso-LR': {'LogisticReg': 0.8548, 'KNN': 0.7983, 'SVM-RBF': 0.8382, 'DecisionTree': 0.8235, 'RandomForest': 0.9051, 'GradBoost': 0.8842, 'XGBoost': 0.9161, 'Stacker': 0.8875},
        'Forward': {'LogisticReg': 0.8923, 'KNN': 0.8445, 'SVM-RBF': 0.8842, 'DecisionTree': 0.7983, 'RandomForest': 0.8824, 'GradBoost': 0.8842, 'XGBoost': 0.8907, 'Stacker': 0.9084},
    },
    'Taiwan': {
        'Full': {'LogisticReg': 0.6257, 'KNN': 0.6399, 'SVM-RBF': 0.6363, 'DecisionTree': 0.6358, 'RandomForest': 0.6703, 'GradBoost': 0.6719, 'XGBoost': 0.6662, 'Stacker': 0.6789},
        'RFE': {'LogisticReg': 0.6143, 'KNN': 0.6167, 'SVM-RBF': 0.6153, 'DecisionTree': 0.6088, 'RandomForest': 0.6751, 'GradBoost': 0.6258, 'XGBoost': 0.6406, 'Stacker': 0.6482},
        'Lasso-LR': {'LogisticReg': 0.6254, 'KNN': 0.6446, 'SVM-RBF': 0.6453, 'DecisionTree': 0.6317, 'RandomForest': 0.6564, 'GradBoost': 0.6683, 'XGBoost': 0.6691, 'Stacker': 0.6801},
        'Forward': {'LogisticReg': 0.6468, 'KNN': 0.6378, 'SVM-RBF': 0.6401, 'DecisionTree': 0.6468, 'RandomForest': 0.6510, 'GradBoost': 0.6207, 'XGBoost': 0.6222, 'Stacker': 0.6362},
    },
}


## 3. Build matrices

In [3]:
def build_matrix(score_dict):
    rows, index = [], []
    for ds in score_dict:
        for fs in FS_ORDER:
            index.append(f'{ds} | {fs}')
            rows.append([score_dict[ds][fs][m] for m in MODELS])
    return pd.DataFrame(rows, index=index, columns=MODELS)

mat_acc   = build_matrix(ACC)
mat_gmean = build_matrix(GMEAN)
print('Accuracy:', mat_acc.shape, '   G-Mean:', mat_gmean.shape)


Accuracy: (16, 8)    G-Mean: (16, 8)


## 4. Wilcoxon + Holm

In [4]:
def wilcoxon_p(a, b):
    if np.all(a - b == 0):
        return 1.0
    return float(wilcoxon(a, b, alternative='two-sided', zero_method='wilcox').pvalue)

def holm_chain(pairs, alpha):
    pairs.sort(key=lambda x: x['p'])
    nc = len(pairs)
    key_sig = f'sig_{alpha:.2f}'
    key_thr = f'thr_{alpha:.2f}'
    rejected = True
    for i, pr in enumerate(pairs):
        thr = alpha / (nc - i)
        pr[key_thr] = thr
        if rejected and pr['p'] < thr:
            pr[key_sig] = True
        else:
            rejected = False
            pr[key_sig] = False
    return pairs

def pairwise(mat):
    pairs = []
    for a, b in combinations(MODELS, 2):
        p = wilcoxon_p(mat[a].values, mat[b].values)
        ma, mb = mat[a].mean(), mat[b].mean()
        winner, loser = (a, b) if ma > mb else ((b, a) if mb > ma else ('tie', 'tie'))
        pairs.append({'a': a, 'b': b, 'mean_a': ma, 'mean_b': mb,
                      'winner': winner, 'loser': loser, 'p': p})
    holm_chain(pairs, 0.05)
    holm_chain(pairs, 0.10)
    return pairs


## 5. Run on Accuracy

In [5]:
pairs_acc = pairwise(mat_acc)
print('All 28 pairs on ACCURACY (sorted by p):')
print(f"  {'pair':<34} {'p':>9} {'thr@0.05':>9} {'sig@0.05':>9} {'thr@0.10':>9} {'sig@0.10':>9}")
for pr in pairs_acc:
    pair = f"{pr['winner']} > {pr['loser']}" if pr['winner']!='tie' else f"{pr['a']} = {pr['b']}"
    sig05 = 'YES' if pr['sig_0.05'] else '.'
    sig10 = 'YES' if pr['sig_0.10'] else '.'
    print(f"  {pair:<34} {pr['p']:>9.4f} {pr['thr_0.05']:>9.4f} {sig05:>9} {pr['thr_0.10']:>9.4f} {sig10:>9}")
print('\nSignificant on ACCURACY at alpha = 0.05:')
for pr in pairs_acc:
    if pr['sig_0.05']:
        print(f"  {pr['winner']} > {pr['loser']}   p = {pr['p']:.4f}")


All 28 pairs on ACCURACY (sorted by p):
  pair                                       p  thr@0.05  sig@0.05  thr@0.10  sig@0.10
  GradBoost > KNN                       0.0008    0.0018       YES    0.0036       YES
  Stacker > KNN                         0.0010    0.0019       YES    0.0037       YES
  XGBoost > KNN                         0.0022    0.0019         .    0.0038       YES
  Stacker > DecisionTree                0.0041    0.0020         .    0.0040         .
  XGBoost > DecisionTree                0.0041    0.0021         .    0.0042         .
  GradBoost > DecisionTree              0.0044    0.0022         .    0.0043         .
  Stacker > LogisticReg                 0.0076    0.0023         .    0.0045         .
  GradBoost > LogisticReg               0.0088    0.0024         .    0.0048         .
  LogisticReg > KNN                     0.0090    0.0025         .    0.0050         .
  Stacker > RandomForest                0.0110    0.0026         .    0.0053         .
  G

## 6. Run on G-Mean

In [6]:
pairs_gmean = pairwise(mat_gmean)
print('All 28 pairs on G-MEAN (sorted by p):')
print(f"  {'pair':<34} {'p':>9} {'thr@0.05':>9} {'sig@0.05':>9} {'thr@0.10':>9} {'sig@0.10':>9}")
for pr in pairs_gmean:
    pair = f"{pr['winner']} > {pr['loser']}" if pr['winner']!='tie' else f"{pr['a']} = {pr['b']}"
    sig05 = 'YES' if pr['sig_0.05'] else '.'
    sig10 = 'YES' if pr['sig_0.10'] else '.'
    print(f"  {pair:<34} {pr['p']:>9.4f} {pr['thr_0.05']:>9.4f} {sig05:>9} {pr['thr_0.10']:>9.4f} {sig10:>9}")
print('\nSignificant on G-MEAN at alpha = 0.05:')
for pr in pairs_gmean:
    if pr['sig_0.05']:
        print(f"  {pr['winner']} > {pr['loser']}   p = {pr['p']:.4f}")


All 28 pairs on G-MEAN (sorted by p):
  pair                                       p  thr@0.05  sig@0.05  thr@0.10  sig@0.10
  Stacker > SVM-RBF                     0.0001    0.0018       YES    0.0036       YES
  Stacker > KNN                         0.0002    0.0019       YES    0.0037       YES
  GradBoost > KNN                       0.0003    0.0019       YES    0.0038       YES
  Stacker > DecisionTree                0.0003    0.0020       YES    0.0040       YES
  XGBoost > KNN                         0.0008    0.0021       YES    0.0042       YES
  Stacker > LogisticReg                 0.0010    0.0022       YES    0.0043       YES
  RandomForest > KNN                    0.0010    0.0023       YES    0.0045       YES
  XGBoost > DecisionTree                0.0063    0.0024         .    0.0048         .
  RandomForest > DecisionTree           0.0110    0.0025         .    0.0050         .
  Stacker > GradBoost                   0.0110    0.0026         .    0.0053         .
  XGB

## 7. Interpretation

A Holm-significant pair $A > B$ at $\alpha$ means we can claim $A$ outperforms $B$
across the 16 blocks, after correcting for 28 simultaneous comparisons.

Compare significant-pair counts on Accuracy vs G-Mean — different metrics may
detect different separations.
